In [1]:
#Soal 1 Data Inspection, Cleaning and Processing

import numpy as np
import pandas as pd

In [2]:
#Load the original CSV into a DataFrame called rawdata
rawdata = pd.read_csv("sales_transactions_raw.csv")

#Display dataset size, memory usage, missing values, unique values, and descriptive statistics
print("Shape:" , rawdata.shape)
rawdata.info(memory_usage="deep")
display(pd.DataFrame({
    "missing" : rawdata.isnull().sum(),
    "unique" : rawdata.nunique(dropna=False),
}))
display(rawdata.describe(include="all").T)

Shape: (7309, 15)
<class 'pandas.DataFrame'>
RangeIndex: 7309 entries, 0 to 7308
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       7309 non-null   str    
 1   order_date     7309 non-null   str    
 2   region         7175 non-null   str    
 3   channel        7309 non-null   str    
 4   customer_id    7309 non-null   str    
 5   customer_type  7309 non-null   str    
 6   sales_rep      7309 non-null   str    
 7   category       7309 non-null   str    
 8   product        7309 non-null   str    
 9   qty            7309 non-null   int64  
 10  discount       7309 non-null   float64
 11  status         7309 non-null   str    
 12  unit_price     7309 non-null   float64
 13  revenue        7309 non-null   float64
 14  cost           6789 non-null   float64
dtypes: float64(4), int64(1), str(10)
memory usage: 4.3 MB


,missing,unique
order_id,0,7159
order_date,0,908
region,134,17
channel,0,5
customer_id,0,2392
customer_type,0,3
sales_rep,0,20
category,0,16
product,0,15
qty,0,97


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,7309,7159,ORD-100005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_date,7309,908,2026-03-18,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,7175,16,Jakarta,1868,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel,7309,5,Offline,4623,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,7309,2392,TEST,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_type,7309,3,Retail,4098,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sales_rep,7309,20,SYS-ONLINE,2686,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,7309,16,Aksesoris,1759,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product,7309,15,Smartphone Lumo A5,662,NaN,NaN,NaN,NaN,NaN,NaN,NaN
qty,7309.0,NaN,NaN,NaN,5.385278,14.765838,-219.0,1.0,3.0,5.0,400.0


In [3]:
#Review categorical columns for inconsistent or unexpected values
for column in ["region","channel","customer_type","sales_rep","category","product","status"]:
    print("\n", column)
    display(rawdata[column].value_counts(dropna=False).to_frame("rows"))

#Create an inspection copy without exact duplicate rows or extra spaces
preview = rawdata.drop_duplicates().copy()
for column in preview.select_dtypes("object"):
    preview[column] = preview[column].str.strip()

# Flag records with potential test or dummy identifiers for inspection
test = (
    preview["order_id"].str.startswith("ORD-T", na=False)
    | preview["customer_id"].eq("TEST")
    | preview["sales_rep"].eq("dummy")
)

#Keeping only real transactions for the next inspection steps
real = preview.loc[~test].copy()

#Show how many exact duplicates and test rows were found
print("Exact duplicates:", rawdata.duplicated().sum())
print("Potential test rows:", test.sum())


 region


,rows
region,
Jakarta,1868
Jawa Barat,1307
Jawa Timur,1219
Sumatera,940
Kalimantan,669
Sulawesi,600
NaN,134
Sumatra,83
Jawa barat,60



 channel


,rows
channel,
Offline,4623
Online,2527
ONLINE,58
online,52
E-Commerce,49



 customer_type


...

,rows
product,
Smartphone Lumo A5,662
Smartphone Lumo X,600
Kertas A4 80gsm (rim),594
Kursi Ergonomis E1,572
Keyboard Mekanik K7,500
Tinta Botol Set CMYK,475
Headset USB H3,465
Laptop Axio Pro 15,463
Laptop Axio 14,457



 status


,rows
status,
Completed,6957
Cancelled,217
Returned,135


Exact duplicates: 150
Potential test rows: 32


/var/folders/0k/87wmm7ts6md1v980wrrz1yf00000gn/T/ipykernel_30677/325967141.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in preview.select_dtypes("object"):


In [4]:
#Parse two date formats observed in the dataset
date_ymd = pd.to_datetime(real["order_date"], format="%Y-%m-%d", errors="coerce")
date_dmy = pd.to_datetime(real["order_date"], format="%d/%m/%Y", errors="coerce")

#Standardize discount scale and calculate expected revenue
discount_check = real["discount"].copy()
discount_check.loc[discount_check > 1] /= 100
expected_revenue = real["qty"] * real["unit_price"] * (1 - discount_check)

#Find prices that may be stored at one-thousandth of the normal scale
normal_price = real.groupby("product")["unit_price"].transform("max")
price_scale_issue = np.isclose(
    real["unit_price"] * 1000,
    normal_price,
    rtol = 0,
    atol = 0.1
)

#Check whether quantity signs aligned with the transaction status
returned = real["status"] == "Returned"
status_quantity_issue = (
    (returned & (real["qty"] >= 0 )) | (~returned & (real["qty"] <= 0))
)

#Check whether online and offline sales use the expected sales representative
online = real["channel"].str.lower().isin(["online","ecommerce"])
sales_rep_issue = (
    (online & (real["sales_rep"] != "SYS-ONLINE")) | (~online & (real["sales_rep"] == "SYS-ONLINE"))
)

#Count every issue so the cleaning decisions supported by evidence
issues = pd.Series({
    "Duplicate order IDs after removal" : real["order_id"].duplicated().sum(),
    "Invalid order ID format" : (~real["order_id"].str.match(r"^(ORD|RET)-\d+$")).sum(),
    "Unparsed dates" : date_ymd.fillna(date_dmy).isna().sum(),
    "Discount above 1" : real["discount"].gt(1).sum(),
    "Possible 1/1000 unit price" : price_scale_issue.sum(),
    "Missing Region" : real["region"].isna().sum(),
    "Missing cost" : real["cost"].isna().sum(),
    "Revenue Formula Mismatch" : (~np.isclose(
        real["revenue"], expected_revenue, rtol = 0, atol = 0.1
    )).sum(),
    "Status and Quantity mismatch" : status_quantity_issue.sum(),
    "Channel and Sales representative mismatch" : sales_rep_issue.sum()
})

display(issues.to_frame("rows"))

,rows
Duplicate order IDs after removal,0
Invalid order ID format,0
Unparsed dates,0
Discount above 1,293
Possible 1/1000 unit price,982
Missing Region,131
Missing cost,506
Revenue Formula Mismatch,0
Status and Quantity mismatch,0
Channel and Sales representative mismatch,48


## Cleaning Decisions

1. Remove the 150 extra exact duplicate copies using all columns, keeping the first occurence. Exclude the 32 confirmed test rows identified by explicit test or dummy markers. Perform both steps before modifying values and record the removed rows in the audit trail. The risk is that an identical record or tes-like identifier represents a legitimate transaction.

2. Parse each observed date format using its explicit format and confirm that every date is valid. Standardize abbreviations only when another column confirms their meaning. Preserve the original values for audit purposes.

3. Convert discounts above 1 to fractions only when the reported revenue confirms that they represent whole percentages. Apply a factor of 1000 only when a row's unit price is exactly one-thousandth of the validated reference price for that product, and apply the same factor to revenue and cost. Accept coreections only if the revenue calculation reconciles afterward.

4. Preserve missing regions, add a missing-region flag, and display them as Unknown only in reporting. Calculate product median unit cost from valid, completed, correctyl scaled transactions. Use it to estimate missing costs, preserve the quantity sign for returns, and add a cost_imputed flag. Clearly disclose products where imputed costs represent a large share of records.

5. Preservce original revenue and cost. Create net metrics that retain the existing negative values for returns and assign zero to cancelled transactions. Keep cancelled rows available for audit or cancellation analysis but exclude them from real-sales KPIs. Flag anomalies using documented rules and report their impact rather than automatically deleting them.

In [5]:
#Start an audit trail that records row counts and total revenue
audit = []

def log(step) :
    audit.append([step, len(df), df["revenue"].sum()])

#Copy the raw data so it remains unchanged
df = rawdata.copy()
log("Raw")

#Save and remove exact duplicate copies before changing any values
duplicate_mask = df.duplicated(keep="first")
duplicate_rows = df.loc[duplicate_mask].copy()

df = df.loc[~duplicate_mask].copy()
log("Remove exact duplicates")

#Identify test rows without modifying the original text values
test_order = df["order_id"].str.strip().str.startswith("ORD-T", na = False)
test_customer = df["customer_id"].str.strip().eq("TEST")
test_sales_rep = df["sales_rep"].str.strip().eq("dummy")

test = test_order | test_customer | test_sales_rep
test_rows = df.loc[test].copy()

print("ORD-T rows :", test_order.sum())
print("TEST customer rows :", test_customer.sum())
print("Dummy sales rep :", test_sales_rep.sum())
print("All test rows :", test.sum())

display(test_rows[["order_id", "customer_id", "sales_rep"]])

df = df.loc[~test].copy()
log("Remove all confirmed test rows")

#Preserve original values before standardizing them
df["order_date_original"] = df["order_date"]
df["region_original"] = df["region"]
df["channel_original"] = df["channel"]
df["category_original"] = df["category"]

#Remove extra spaces from the working text columns
for column in df.select_dtypes("str").columns:
    df[column] = df[column].str.strip()


#Parse two date formats observed in the dataset
date_ymd = pd.to_datetime(df["order_date"], format="%Y-%m-%d", errors="coerce")
date_dmy = pd.to_datetime(df["order_date"], format="%d/%m/%Y", errors="coerce")

df["order_date"] = date_ymd.fillna(date_dmy)

print("ISO dates:" , date_ymd.notna().sum())
print("DD/MM/YYYY dates:" , (date_ymd.isna() & date_dmy.notna()).sum())
print("Unparsed dates:" , df["order_date"].isna().sum())

assert df["order_date"].notna().all()

#Define the label mappings supported by the inspection
region_map = {
    "Dki Jakarta" : "Jakarta",
    "Jkt" : "Jakarta",
    "Jabar" : "Jawa Barat",
    "Jatim" : "Jawa Timur",
    "Sumatra" : "Sumatera"
}

category_map = {
    "Aksesori" : "Aksesoris",
    "Hp" : "Smartphone",
    "Atk" : "Peralatan Kantor",
    "Printer&Tinta" : "Printer & Tinta"
}

#Standardize the working labels while retaining the original columns
df["region_missing_flag"] = df["region"].isna()

df["region"] = (
    df["region"].str.title().replace(region_map)
)

df["region_reporting"] = df["region"].fillna("Unknown")

df["channel"] = (
    df["channel"].str.title().replace({"E-Commerce" : "Online"})
)

df["category"] = (
    df["category"].str.title().replace(category_map)
)

log("Standardizing dates and labels")

#Display the audit trail
audit_log = pd.DataFrame(
    audit,
    columns=["step","rows","total_revenue"]
)

display(audit_log)

ORD-T rows : 32
TEST customer rows : 32
Dummy sales rep : 32
All test rows : 32


,order_id,customer_id,sales_rep
5346,ORD-T001,TEST,dummy
5347,ORD-T002,TEST,dummy
5348,ORD-T003,TEST,dummy
5361,ORD-T004,TEST,dummy
5362,ORD-T005,TEST,dummy
5363,ORD-T006,TEST,dummy
5378,ORD-T007,TEST,dummy
5379,ORD-T008,TEST,dummy
5380,ORD-T009,TEST,dummy
5398,ORD-T010,TEST,dummy


ISO dates: 6145
DD/MM/YYYY dates: 982
Unparsed dates: 0


,step,rows,total_revenue
0,Raw,7309,5.440126e+10
1,Remove exact duplicates,7159,5.346553e+10
2,Remove all confirmed test rows,7127,5.346553e+10
3,Standardizing dates and labels,7127,5.346553e+10


In [6]:
#Preserve the original numeric values for auditing
df["discount_original"] = df["discount"]
df["unit_price_original"] = df["unit_price"]
df["revenue_original"] = df["revenue"]
df["cost_original"] = df["cost"]

#Identify discounts stored as whole percentages
discount_candidate = df["discount"].gt(1)

#Calculate revenue assuming the discount is a whole percentage
revenue_if_percentage = (
    df["qty"] * df["unit_price"] * (1 - df["discount"]/100)
)

#Correct only discounts confirmed by the reported revenue
df["discount_corrected_flag"] = (
    discount_candidate & np.isclose(
        df["revenue"],
        revenue_if_percentage,
        rtol = 0,
        atol = 0.01
    )
)

#Check whether any discount above 1 remains unconfirmed
unconfirmed_discount = (
    discount_candidate & ~df["discount_corrected_flag"]
)

print("Discounts above 1:" , discount_candidate.sum())
print("Confirmed discount corrections:", df["discount_corrected_flag"].sum())
print("Unconfirmed discounts:",unconfirmed_discount.sum())

#Stop if a discount above 1 is not supported by revenue
assert unconfirmed_discount.sum() == 0

#Convert confirmed whole percentages into fractions
df.loc[
    df["discount_corrected_flag"],
    "discount"
] /= 100

#Find the most common unit price for every product
reference_price = (
    df.groupby("product")["unit_price"].agg(lambda values: values.mode().iloc[0])
)

#Add the reference price to each row based on its product
df["reference_unit_price"] = df["product"].map(reference_price)

#Identify prices stored at exactly one-thousandth of the reference
df["scale_corrected_flag"] = np.isclose(
    df["unit_price"] * 1000,
    df["reference_unit_price"],
    rtol = 0,
    atol = 0.01
)

#Apply the correction only to monetary columns
df.loc[
    df["scale_corrected_flag"],
    ["unit_price","revenue","cost"]
] *= 1000

print("Scale corrections:", df["scale_corrected_flag"].sum())

#Calculate expected revenue after both corrections
expected_revenue = (
    df["qty"] * df["unit_price"] * (1 - df["discount"])
)

#Check whether reported revenue now matches the formula
revenue_matches = np.isclose(
    df["revenue"],
    expected_revenue,
    rtol = 0,
    atol = 0.01
)

print("Revenue mismatches after corrections:" ,(~revenue_matches).sum())

#Stop if any revenue still does not match
assert revenue_matches.all()

#Use the verified calculated revenue as the cleaned revenue
df["revenue"] = expected_revenue

log("Correct discounts prices and revenu")



Discounts above 1: 293
Confirmed discount corrections: 293
Unconfirmed discounts: 0
Scale corrections: 982
Revenue mismatches after corrections: 0


In [7]:
#Use completed transactions with known costs as the reference
completed_with_cost = df[
    (df["status"] == "Completed") & df["cost"].notna() & df["qty"].gt(0)
].copy()

#Calculate the observed unit cost
completed_with_cost["unit_cost"] = (
    completed_with_cost["cost"] / completed_with_cost["qty"]
)

#Summarize cost variation for every product
cost_stability = (
    completed_with_cost.groupby("product")["unit_cost"]
    .agg(["count", "min", "median","max"])
)

cost_stability["range_pct"] = (
    (cost_stability["max"] - cost_stability["min"]) / cost_stability["median"] * 100
)

display(cost_stability.sort_values("range_pct", ascending=False))

#Flag rows whose original cost is missing
df["cost_imputed_flag"] = df["cost"].isna()

#Calculate the median completed unit cost for every product
median_unit_cost = (
    completed_with_cost.groupby("product")["unit_cost"].median()
)

#Identify rows that need a cost estimate
missing_cost = df["cost"].isna()

#Match every missing row to its product's median unit cost
estimated_unit_cost = (
    df.loc[missing_cost, "product"].map(median_unit_cost)
)

#Confirm that every missing row has an available estimate
assert estimated_unit_cost.notna().all()

#Insert the estimated costs into the DataFrame
df.loc[missing_cost, "cost"] = (
    df.loc[missing_cost, "qty"]
    * estimated_unit_cost
)

#Confirm that imputed return costs remain negative
returned_imputed = (
    df["cost_imputed_flag"] & df["status"].eq("Returned")
)

returned_costs_negative = (
    df.loc[returned_imputed, "cost"].lt(0).all()
)

print("Imputed costs:", df["cost_imputed_flag"].sum())
print("Missing costs after imputation:", df["cost"].isna().sum())
print("Returned imputed costs remain negative:", returned_costs_negative)

assert returned_costs_negative

log("Impute missing cost")

#Calculate the percentage of imputed costs for every product
imputation_summary = (
    df.groupby("product", as_index = False)
    .agg(
        rows = ("order_id", "size"),
        imputed_rows = ("cost_imputed_flag", "sum")
    )
)

imputation_summary["imputed_pct"] = (
    imputation_summary["imputed_rows"] 
    / imputation_summary["rows"]
    *100
)

imputation_summary = imputation_summary.sort_values(
    "imputed_pct",
    ascending = False
)

#Display the final step 8 evidence
audit_log = pd.DataFrame(
    audit,
    columns = ["step","rows","total_revenue"]
)

display(audit_log)
display(imputation_summary)

print("Discount corrections:", df["discount_corrected_flag"].sum())
print("Scale corrections:", df["scale_corrected_flag"].sum())
print("Imputed costs:", df["cost_imputed_flag"].sum())

,count,min,median,max,range_pct
product,,,,,
Kertas A4 80gsm (rim),552,37400.0,38100.0,38600.0,3.149606
Mouse Wireless M20,406,103400.0,104900.0,106600.0,3.050524
Tinta Botol Set CMYK,440,162500.0,164900.0,167500.0,3.032141
Power Bank 20000mAh,405,197000.0,200000.0,203000.0,3.000000
Laptop Axio 14,421,7486300.0,7593600.0,7714000.0,2.998578
Printer InkJet P200,420,1674600.0,1699150.0,1725400.0,2.989730
Smartphone Lumo X,553,4728500.0,4799300.0,4871800.0,2.985852
Laptop Axio Pro 15,432,11477300.0,11644350.0,11824200.0,2.979127
Laptop Veno 13,373,5467200.0,5549600.0,5632300.0,2.974989


Imputed costs: 506
Missing costs after imputation: 0
Returned imputed costs remain negative: True


,step,rows,total_revenue
0,Raw,7309,5.440126e+10
1,Remove exact duplicates,7159,5.346553e+10
2,Remove all confirmed test rows,7127,5.346553e+10
3,Standardizing dates and labels,7127,5.346553e+10
4,Correct discounts prices and revenu,7127,6.290626e+10
5,Impute missing cost,7127,6.290626e+10


,product,rows,imputed_rows,imputed_pct
13,Switch 8 Port S8,382,234,61.256545
10,Router WiFi 6 R600,377,221,58.620690
9,Printer InkJet P200,441,7,1.587302
2,Keyboard Mekanik K7,489,5,1.022495
6,Laptop Veno 13,398,4,1.005025
8,Power Bank 20000mAh,428,4,0.934579
7,Mouse Wireless M20,433,4,0.923788
14,Tinta Botol Set CMYK,464,4,0.862069
12,Smartphone Lumo X,585,5,0.854701
3,Kursi Ergonomis E1,558,4,0.716846


Discount corrections: 293
Scale corrections: 982
Imputed costs: 506


In [8]:
#Create date fields for monthly and quarterly analysis
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["quarter"] = "Q" + df["order_date"].dt.quarter.astype(str)
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)

#Identify cancelled transactions
cancelled = df["status"].eq("Cancelled")

#Preserve cleaned revenue and cost in separate net fields
df["net_revenue"] = df["revenue"]
df["net_cost"] = df["cost"]

#Exclude cancelled transactions from sales KPIs
df.loc[
    cancelled,
    ["net_revenue","net_cost"]
] = 0

#Calculate gross profit and margin
df["gross_profit"] = (
    df["net_revenue"] - df["net_cost"]
)

df["margin_pct"] = (
    df["gross_profit"] / df["net_revenue"].replace(0, np.nan)
)

display(
    df.loc[
        cancelled,
        [
            "order_id",
            "status",
            "revenue",
            "cost",
            "net_revenue",
            "net_cost",
        ]
    ].head()
)

,order_id,status,revenue,cost,net_revenue,net_cost
73,ORD-100021,Cancelled,13000000.0,11794700.0,0.0,0.0
122,ORD-100519,Cancelled,110000.0,75800.0,0.0,0.0
189,ORD-100275,Cancelled,360000.0,211800.0,0.0,0.0
195,ORD-100534,Cancelled,2660000.0,2416400.0,0.0,0.0
207,ORD-100042,Cancelled,660000.0,452400.0,0.0,0.0


In [9]:
#Select completed transactions for return matching
completed_orders = df.loc[
    df["status"].eq("Completed"),
    ["order_id", "product", "qty", "revenue"]
].copy()


#Select returned transactions
returned_orders = df.loc[
    df["status"].eq("Returned"),
    ["order_id", "product", "qty", "revenue", "cost"]
].copy()


#Convert each return ID into its expected original order ID
returned_orders["original_order_id"] = (
    returned_orders["order_id"]
    .str.replace("RET-", "ORD-", regex=False)
)


#Match each return with its original completed order
return_check = returned_orders.merge(
    completed_orders,
    left_on="original_order_id",
    right_on="order_id",
    how="left",
    suffixes=("_return", "_completed"),
)


#Check the matched order, product, quantity, and revenue
return_check["id_match"] = (
    return_check["order_id_completed"].notna()
)

return_check["product_match"] = (
    return_check["product_return"]
    == return_check["product_completed"]
)

return_check["quantity_reversed"] = (
    return_check["qty_return"]
    == -return_check["qty_completed"]
)

return_check["revenue_reversed"] = np.isclose(
    return_check["revenue_return"],
    -return_check["revenue_completed"],
    rtol=0,
    atol=0.01,
)


# Create the final validation table
validation = pd.Series({
    "order IDs unique after cleaning":
        df["order_id"].is_unique,

    "dates parsed":
        df["order_date"].notna().all(),

    "reporting regions complete":
        df["region_reporting"].notna().all(),

    "discounts between 0 and 1":
        df["discount"].between(0, 1).all(),

    "cost complete":
        df["cost"].notna().all(),

    "revenue formula correct":
        np.isclose(
            df["revenue"],
            df["qty"]
            * df["unit_price"]
            * (1 - df["discount"]),
            rtol=0,
            atol=0.01,
        ).all(),

    "return values negative":
        returned_orders[
            ["qty", "revenue", "cost"]
        ].lt(0).all().all(),

    "return IDs matched":
        return_check["id_match"].all(),

    "return products matched":
        return_check["product_match"].all(),

    "return quantities reversed":
        return_check["quantity_reversed"].all(),

    "return revenues reversed":
        return_check["revenue_reversed"].all(),

    "cancelled net values zero":
        df.loc[
            cancelled,
            ["net_revenue", "net_cost"]
        ].eq(0).all().all(),

    "non-cancelled net values retained":
        (
            np.isclose(
                df.loc[~cancelled, "net_revenue"],
                df.loc[~cancelled, "revenue"],
            ).all()
            and
            np.isclose(
                df.loc[~cancelled, "net_cost"],
                df.loc[~cancelled, "cost"],
            ).all()
        ),
})


#Display the validation results
display(validation.to_frame("passed"))

#Stop if any validation result fails
assert validation.all()

,passed
order IDs unique after cleaning,True
dates parsed,True
reporting regions complete,True
discounts between 0 and 1,True
cost complete,True
revenue formula correct,True
return values negative,True
return IDs matched,True
return products matched,True
return quantities reversed,True


In [10]:
#Calculate the overall margin using all cleaned rows
margin_with_imputation = (
    df["gross_profit"].sum() / df["net_revenue"].sum()
)

#Select transactions whose costs were originally available
known_cost_rows = -df["cost_imputed_flag"]

#Calculate margin using only rows with original known costs
margin_known_cost_only = (
    df.loc[known_cost_rows,"gross_profit"].sum() / df.loc[known_cost_rows, "net_revenue"].sum()
)

#Display the sensitivity comparison
print(
    "Overall margin with imputed costs:", f"{margin_with_imputation:.2%}"
)

print(
    "Margin using known-cost rows only:", f"{margin_known_cost_only:.2%}"
)

Overall margin with imputed costs: 13.08%
Margin using known-cost rows only: 12.56%


## Required Summary Tables

1. Monthly net revenue, gross profit, and margin
2. Net revenue and margin by region and category
3. Top 10 customers by net revenue

Cancelled transactions contribute zero to the summaries, while returned transactions retain their negative values.

In [11]:
#Summarize performance for every month
monthly = (
    df.groupby("year_month", as_index = False)
    [["net_revenue", "gross_profit"]].sum()
)

#Calculate monthly margin from the monthly totals
monthly["margin_pct"] = (
    monthly["gross_profit"] / monthly["net_revenue"].replace(0, np.nan)
)

#Display the monthly summary

display(monthly)

,year_month,net_revenue,gross_profit,margin_pct
0,2024-09,1.798231e+09,285066800.0,0.158526
1,2024-10,3.477732e+09,439519500.0,0.126381
2,2024-11,2.222648e+09,349582100.0,0.157282
3,2024-12,2.558929e+09,352981750.0,0.137941
4,2025-01,1.577241e+09,236160450.0,0.149730
5,2025-02,1.969671e+09,293955050.0,0.149241
6,2025-03,2.813316e+09,454806450.0,0.161662
7,2025-04,1.763232e+09,255900350.0,0.145131
8,2025-05,2.034898e+09,291040650.0,0.143025
9,2025-06,2.197033e+09,314944000.0,0.143350


In [12]:
#Summarize performance for every region and category
region_category = (
    df.groupby(
        ["region_reporting", "category"], as_index = False
    )[["net_revenue", "gross_profit"]].sum()
)

#Calculate margin for every region and category
region_category["margin_pct"] = (
    region_category["gross_profit"]
    / region_category["net_revenue"].replace(0, np.nan)
)

#Sort the table by region and revenue
region_category = region_category.sort_values(
    ["region_reporting", "net_revenue"],
    ascending = [True, False]
)

#Display the region and category summary

display(region_category)

,region_reporting,category,net_revenue,gross_profit,margin_pct
1,Jakarta,Laptop,7.991445e+09,561416650.0,0.070252
5,Jakarta,Smartphone,3.429200e+09,252802800.0,0.073721
3,Jakarta,Peralatan Kantor,1.969826e+09,553265550.0,0.280870
2,Jakarta,Networking,1.380814e+09,361842500.0,0.262050
0,Jakarta,Aksesoris,1.288279e+09,456655400.0,0.354469
4,Jakarta,Printer & Tinta,6.675242e+08,131916350.0,0.197620
7,Jawa Barat,Laptop,5.982430e+09,332243900.0,0.055537
11,Jawa Barat,Smartphone,2.184085e+09,179219600.0,0.082057
10,Jawa Barat,Printer & Tinta,1.389220e+09,208502900.0,0.150086
9,Jawa Barat,Peralatan Kantor,1.151744e+09,326934900.0,0.283861


In [13]:
#Summarize performance for every customer

top_customers = (
    df.groupby("customer_id", as_index = False)
    [["net_revenue", "gross_profit"]].sum()
)

#Calculate each customer's margin
top_customers["margin_pct"] = (
    top_customers["gross_profit"] / top_customers["net_revenue"].replace(0, np.nan)
)

#Select the ten customers with the highest net revenue
top_customers = (
    top_customers.sort_values("net_revenue", ascending = False)
    .head(10)
)

#Display the top ten customers
display(top_customers)

,customer_id,net_revenue,gross_profit,margin_pct
1769,C01919,1.708595e+09,184234400.0,0.107828
1564,C01695,6.330400e+08,70876800.0,0.111963
2187,C02374,5.080000e+08,21646900.0,0.042612
2313,K0102,5.042520e+08,66033000.0,0.130952
1611,C01747,5.033000e+08,-15213200.0,-0.030227
677,C00737,5.002250e+08,20596700.0,0.041175
2384,K0173,4.819210e+08,48873600.0,0.101414
2338,K0127,3.842695e+08,39697900.0,0.103307
2242,K0031,3.759760e+08,76105500.0,0.202421
2362,K0151,3.640050e+08,33982200.0,0.093356


In [14]:
#Calculate the final overall totals
total_net_revenue = df["net_revenue"].sum()
total_gross_profit = df["gross_profit"].sum()
overall_margin = total_gross_profit / total_net_revenue

#Display final totals
print("Final rows:", len(df))
print("Net Revenue:", total_net_revenue)
print("Gross Profit:", total_gross_profit)
print("Overall Margin:", overall_margin)

Final rows: 7127
Net Revenue: 61172726150.0
Gross Profit: 8002284550.0
Overall Margin: 0.1308145811644525


## Anomalies, Findings, and Recommendations

This section flags unusual quantities and negative-margin transactions. These records are retained because an unusual transaction is not automatically an error.

In [15]:
#Use absolute quantity so returns are checked correctly
absolute_qty = df["qty"].abs()

#Calculate the IQR limit for each customer type
q1 = absolute_qty.groupby(
    df["customer_type"]
).transform("quantile", q = 0.25)

q3 = absolute_qty.groupby(
    df["customer_type"]
).transform("quantile", q = 0.75)

iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr

#Flag unusual quantities and negative margins
df["qty_outlier_flag"] = absolute_qty > upper_limit
df["negative_margin_flag"] = df["margin_pct"] < 0

#Display the number of flagged rows
print("Quantity outliers:", df["qty_outlier_flag"].sum())
print("Negative margin rows:", df["negative_margin_flag"].sum())

Quantity outliers: 36
Negative margin rows: 395


In [16]:
#Prepare quantity outlier examples
quantity_examples = df.loc[
    df["qty_outlier_flag"],
    ["order_id", "customer_type", "product", "qty", "net_revenue"]
].copy()

quantity_examples["absolute_qty"] = quantity_examples["qty"].abs()

#Display the largest quantity outliers
display(
    quantity_examples
    .sort_values("absolute_qty", ascending = False)
    .head(10)
)

#Display the largest negative margin transactions

display(
    df.loc[
        df["negative_margin_flag"],
        [
            "order_id",
            "status",
            "product",
            "discount",
            "net_revenue",
            "gross_profit"
        ]
    ]
    .sort_values("gross_profit")
    .head(10)
)


,order_id,customer_type,product,qty,net_revenue,absolute_qty
5678,ORD-105606,Retail,Switch 8 Port S8,400,180000000.0,400
3097,ORD-102950,Korporat,Headset USB H3,289,91035000.0,289
2623,ORD-103803,Korporat,Tinta Botol Set CMYK,273,67267200.0,273
4105,ORD-103891,Korporat,Router WiFi 6 R600,272,300288000.0,272
3370,ORD-103197,Korporat,Printer InkJet P200,261,504252000.0,261
3603,ORD-103387,Korporat,Keyboard Mekanik K7,248,148304000.0,248
896,ORD-100758,Korporat,Keyboard Mekanik K7,246,143910000.0,246
943,ORD-100899,Korporat,Keyboard Mekanik K7,242,138424000.0,242
6574,ORD-106438,Korporat,Keyboard Mekanik K7,220,125840000.0,220
2271,ORD-102458,Korporat,Kertas A4 80gsm (rim),219,10840500.0,219


,order_id,status,product,discount,net_revenue,gross_profit
6578,ORD-106442,Completed,Laptop Axio Pro 15,0.30,91000000.0,-24804000.0
3197,ORD-103287,Completed,Laptop Axio Pro 15,0.30,72800000.0,-20967200.0
6229,ORD-106118,Completed,Smartphone Lumo X,0.15,467500000.0,-16460000.0
4854,ORD-104696,Completed,Laptop Axio Pro 15,0.30,45500000.0,-13075500.0
7199,ORD-106286,Completed,Laptop Axio Pro 15,0.25,48750000.0,-9936500.0
4880,ORD-104719,Completed,Laptop Axio Pro 15,0.30,36400000.0,-9863200.0
6425,ORD-106287,Completed,Laptop Axio Pro 15,0.25,48750000.0,-9637000.0
6910,ORD-106731,Completed,Laptop Axio Pro 15,0.30,36400000.0,-9539600.0
3807,ORD-104182,Completed,Laptop Axio Pro 15,0.25,48750000.0,-9213500.0
5331,ORD-105327,Completed,Laptop Axio Pro 15,0.25,48750000.0,-9026500.0


In [17]:
#Summarize product performance
product = (
    df.groupby("product", as_index = False)
    [["net_revenue", "gross_profit"]].sum()
)

#Calculate product margin
product["margin_pct"] = (
    product["gross_profit"]
    /product["net_revenue"].replace(0, np.nan)
)

#Rank products by net revenue
product = product.sort_values(
    "net_revenue",
    ascending = False
)

display(product.head(10))

,product,net_revenue,gross_profit,margin_pct
5,Laptop Axio Pro 15,1.282060e+10,6.255246e+08,0.048791
4,Laptop Axio 14,9.584600e+09,6.276797e+08,0.065488
12,Smartphone Lumo X,8.154300e+09,5.372096e+08,0.065881
3,Kursi Ergonomis E1,7.333639e+09,1.999539e+09,0.272653
6,Laptop Veno 13,6.108550e+09,3.915474e+08,0.064098
11,Smartphone Lumo A5,3.818080e+09,2.894908e+08,0.075821
10,Router WiFi 6 R600,3.231132e+09,7.769640e+08,0.240462
9,Printer InkJet P200,3.141999e+09,4.055432e+08,0.129072
2,Keyboard Mekanik K7,2.453926e+09,8.328436e+08,0.339392
13,Switch 8 Port S8,1.244606e+09,3.679986e+08,0.295675


In [18]:
#Summarize performance by reporting region
region = (
    df.groupby("region_reporting", as_index = False)
    [["net_revenue" , "gross_profit"]].sum()
)

#Calculate region margin
region["margin_pct"] = (
    region["gross_profit"]
    / region["net_revenue"].replace(0, np.nan)
)

#Rank regions by gross profit
region = region.sort_values(
    "gross_profit",
    ascending = False
)

display(region)

,region_reporting,net_revenue,gross_profit,margin_pct
0,Jakarta,1.672709e+10,2.317899e+09,0.138572
1,Jawa Barat,1.233427e+10,1.550964e+09,0.125744
2,Jawa Timur,1.038902e+10,1.185753e+09,0.114135
5,Sumatera,8.929370e+09,1.137056e+09,0.127339
4,Sulawesi,5.776738e+09,8.721950e+08,0.150984
3,Kalimantan,6.155725e+09,8.165851e+08,0.132655
6,Unknown,8.605170e+08,1.218330e+08,0.141581


## Findings and Recommendations

### Findings

1. Laptop Axio Pro 15 generated the highest net revenue at approximately 12.82 billion.
2. Kursi Ergonomis E1 generated the highest gross profit at approximately 2.00 billion.
3. Jakarta generated the highest regiuonal net revenue and gross profit at approximately 16.73 billion and 2.32 billion.
4. The analysis flagged 36 quantity outliers and 395 negative margin transactions.

### Recommendations

1. Ask the Sales Operations teams to review the flagged quantity and negative margin transactions.
2. Add automated checks for duplicates, date formats, discount scales, monetary scales, and revenue calculations.
3. Require product cost and clearer regional information for future transactions.

### Limitations

- The 506 imputed costs are estimation based on the median compelted unit cost for each product
- The 131 missing regions are displayed as 'Unknown'.
- An anomaly flag means that a transactions is unsual, not necessarily incorrect. 